<center> <h2> Using Kafkanator Fairness metrics table  </h2> </center>

<p> In this notebook you will learn to use kafkanator create_dataviz method to quickly get a summary of fairness metrics in your predictive model.
For this, we will use the ADULT dataset, a well known dataset in fairness domain to predict income based in some attributes, including sensistive ones such as gender and race. </p>
<h3> Outline : </h3>
<ol>
    <li> Data Downloading </li>
    <li> Data Preprocessing </li>
    <li> Training Model </li>
    <li> Executing the fairness metrics table method .</li>
    <ol>
        <li>A single sensitive attribute . </li>
        <li>Multiple sensitive attributes. </li>
</ol>

In [1]:
%pip install ucimlrepo


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


<h3> 1. Data Downloading </h3>

In [1]:
from ucimlrepo import fetch_ucirepo 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from kafkanator import *
from collections import Counter
import xgboost as xgb
import pandas as pd
import numpy as np
from kafkanator.fairness import *
import kafkanator.util as ut

In [3]:
# fetch dataset 
adult = fetch_ucirepo(id=2) 
# data (as pandas dataframes) 
data = adult.data.features 
y = adult.data.targets 
# metadata 
print(adult.metadata) 
# variable information 
print(adult.variables) 
# Data cleaning
y['income'] = y['income'].apply(lambda x: x.replace('.',''))
data['sex'] = data['sex'].str.strip()
data['race'] = data['race'].str.strip()
data['marital-status'] = data['marital-status'].str.strip()
data['native-country'] = data['native-country'].str.strip()

{'uci_id': 2, 'name': 'Adult', 'repository_url': 'https://archive.ics.uci.edu/dataset/2/adult', 'data_url': 'https://archive.ics.uci.edu/static/public/2/data.csv', 'abstract': 'Predict whether annual income of an individual exceeds $50K/yr based on census data. Also known as "Census Income" dataset. ', 'area': 'Social Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 48842, 'num_features': 14, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Income', 'Education Level', 'Other', 'Race', 'Sex'], 'target_col': ['income'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1996, 'last_updated': 'Tue Sep 24 2024', 'dataset_doi': '10.24432/C5XW20', 'creators': ['Barry Becker', 'Ronny Kohavi'], 'intro_paper': None, 'additional_info': {'summary': "Extraction was done by Barry Becker from the 1994 Census database.  A set of reasonably clean records was extracted using the fol

In [4]:
data

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48837,39,Private,215419,Bachelors,13,Divorced,Prof-specialty,Not-in-family,White,Female,0,0,36,United-States
48838,64,NaN,321403,HS-grad,9,Widowed,NaN,Other-relative,Black,Male,0,0,40,United-States
48839,38,Private,374983,Bachelors,13,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,50,United-States
48840,44,Private,83891,Bachelors,13,Divorced,Adm-clerical,Own-child,Asian-Pac-Islander,Male,5455,0,40,United-States


<h3> 2. Data preprocessing </h3>

In [5]:
data['race'] = data['race'].apply(lambda x : 1 if x=='White' else 0 )

In [6]:
data[data['sex']=='Male'].shape

(32650, 14)

In [7]:
data[data['sex']=='Female'].shape

(16192, 14)

In [8]:
categorical_columns = ['workclass', 'education',  'occupation', 'relationship', 'race', 'sex','marital-status','native-country']

# Create an instance of LabelEncoder
label_encoder = LabelEncoder()

# Encode each categorical column and create new feature columns
for col in categorical_columns:
    encoded_col_name = f"{col}_encoded"
    data[encoded_col_name] = label_encoder.fit_transform(data[col])

In [9]:
Y = label_encoder.fit_transform(y['income'])

In [10]:
X = data[['age','workclass_encoded', 'education_encoded','education-num','marital-status_encoded','occupation_encoded',	'relationship_encoded','race_encoded','sex_encoded','capital-loss','capital-gain','hours-per-week','native-country_encoded']]

<h2> 3. Training the model </h2>

In [11]:
X_train, X_test, y_train, y_test = train_test_split( X, Y, test_size=0.3)

In [12]:
#classifier = RandomForestClassifier(n_estimators=100, random_state=42)
classifier = xgb.XGBClassifier(objective="binary:logistic")
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)

In [13]:
y_pred

array([0, 0, 1, ..., 0, 0, 0])

In [14]:
sens_metrics = X_test['sex_encoded']

In [15]:
X_test

,age,workclass_encoded,education_encoded,education-num,marital-status_encoded,occupation_encoded,relationship_encoded,race_encoded,sex_encoded,capital-loss,capital-gain,hours-per-week,native-country_encoded
3112,24,4,11,9,4,1,3,1,1,0,0,40,39
27147,41,7,11,9,4,8,1,1,0,0,0,40,39
35323,71,4,8,11,6,10,4,1,0,0,11678,38,39
25971,55,4,11,9,6,7,4,0,0,0,0,44,39
2011,23,4,15,10,4,8,3,1,0,0,0,20,39
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9265,59,6,11,9,2,12,0,1,1,0,0,40,9
19644,50,4,11,9,2,3,0,1,1,0,0,40,39
16860,24,4,11,9,4,8,2,1,1,0,0,25,39
8181,20,4,7,12,4,8,3,1,0,2205,0,18,39


<h2> 4. Executing the fairness metrics table method </h2>
<p> This method receives as parameter a numpy matrix with sensitive attribute, prediction and reality columns. You can specify
one sensitive attribute in an array, or multiple. As indicated in followint subsections </p>

In [16]:
fairness_analysis = np.transpose(np.array([X_test['sex_encoded'].values, y_pred,y_test]))
fairness_analysis_df = pd.DataFrame(data=fairness_analysis,columns=['sex_encoded','prediction','reality'])

<h2> 4.1. One sensitive attribute : Gender </h2>

In [17]:
sp = fairness_metrics_table(fairness_analysis_df,['sex_encoded'],'prediction','reality',aggregate_metrics=False)

ks  {'0': 0.08616455178059762, '1': 0.2626190232415276}
 lcosl  ['0', '1']
sta party output  {(0,): 0.08616455178059762, (1,): 0.2626190232415276}


In [18]:
sp

,0,1
DEMOGRAPHIC PARITY - P1,0.086165,0.262619
EQUAL OPPORTUNITY - TPR,0.588235,0.669045
PREDICTIVE PARITY - PPV,0.736342,0.767641
DISPARATE IMPACT - PREVALENCE,0.107859,0.301321


In [19]:
sp.style.apply(ut.default_row_highlighting,axis=1)

d is  0.17645447146092996
 in if
d is  0.08080989786332471
 in if
d is  0.03129928278071792
 in if
d is  0.19346158451393525
 in if


,0,1
DEMOGRAPHIC PARITY - P1,0.086165,0.262619
EQUAL OPPORTUNITY - TPR,0.588235,0.669045
PREDICTIVE PARITY - PPV,0.736342,0.767641
DISPARATE IMPACT - PREVALENCE,0.107859,0.301321


<h2> You can also obtain normal fairness metrics by using kafkanator fairness module methods: </h2>

In [23]:
eo = equal_opportunity(concatenated,'sex_encoded','prediction','reality')
eod = equalized_odds(concatenated,'sex_encoded','prediction','reality')
pp = predictive_parity(concatenated,'sex_encoded','prediction','reality')

In [24]:
eo

{0: 0.6095617529880478, 1: 0.6773636991028296}

In [25]:
eod

{0: '0.023304107060452238,0.6095617529880478',
 1: '0.08151466974996387,0.6773636991028296'}

In [26]:
pp

{0: 0.7518427518427518, 1: 0.776810447170558}

<h3> 4.2 Multiple sensitive attributes : gender and race </h3>
<p> Just build a numpy matrix containing sensitive attributes, reality and predictions columns, then pass an array containning
an arbitrary set of sensitive attributes </p>

In [22]:
fairness_analysis = np.transpose(np.array([X_test['sex_encoded'].values,X_test['race_encoded'].values, y_pred,y_test]))
concatenated = pd.DataFrame(data=fairness_analysis,columns=['sex_encoded','race_encoded','prediction','reality'])
sp = fairness_metrics_table(concatenated,['sex_encoded','race_encoded'],'prediction','reality',aggregate_metrics=False)

ks  {'0,0': 0.04951560818083961, '0,1': 0.09476876421531463, '1,0': 0.1759656652360515, '1,1': 0.27435480120902117}
 lcosl  ['0,0', '0,1', '1,0', '1,1']
sta party output  {(0, 0): 0.04951560818083961, (0, 1): 0.09476876421531463, (1, 0): 0.1759656652360515, (1, 1): 0.27435480120902117}


In [23]:
sp

,"0,0","0,1","1,0","1,1"
DEMOGRAPHIC PARITY - P1,0.049516,0.094769,0.175966,0.274355
EQUAL OPPORTUNITY - TPR,0.415584,0.617778,0.639344,0.67173
PREDICTIVE PARITY - PPV,0.695652,0.741333,0.760976,0.76822
DISPARATE IMPACT - PREVALENCE,0.082885,0.113723,0.209442,0.313764
